# 05. Fixed Income Risk & Scenario Analysis

This notebook evaluates the interest-rate and credit-spread sensitivity of the fixed-income universe and frozen regime portfolios.

The analysis includes:

- Macaulay and modified duration
- Effective duration
- Convexity
- DV01
- Rate-shock scenarios
- Credit-spread shock scenarios
- Portfolio-level risk aggregation

ETF portfolio characteristics use a current iShares risk snapshot, while credit-spread sensitivity is explicitly treated as a first-order proxy rather than an exact bond-level repricing model.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

REGIME_PATH = (
    PROCESSED_DIR
    / "fixed_income_regime_panel.csv"
)

regime_data = pd.read_csv(
    REGIME_PATH,
    index_col="Date",
    parse_dates=True,
)

regime_data.index.name = "Date"

print("Project root :", PROJECT_ROOT)

print(
    "Regime sample:",
    regime_data.index.min(),
    "to",
    regime_data.index.max()
)

Project root : c:\Users\minha\Desktop\global-fixed-income-strategy
Regime sample: 2008-01-31 00:00:00 to 2026-07-31 00:00:00


## 1. ETF Risk Snapshot

The following portfolio characteristics are taken from official iShares fund data.

**Risk snapshot date:** August 27, 2026

Effective duration is used for first-order interest-rate sensitivity.

For LQD and HYG, effective duration is also used as a simplified spread-duration proxy because a separate spread-duration measure is not provided in the public fund characteristics. Credit scenario results should therefore be interpreted as sensitivity estimates rather than exact repricing forecasts.

In [3]:
ETF_TICKERS = [
    "SHY",
    "IEF",
    "TLT",
    "LQD",
    "HYG",
]

risk_snapshot = pd.DataFrame(
    {
        "Effective_Duration": {
            "SHY": 1.78,
            "IEF": 6.82,
            "TLT": 14.95,
            "LQD": 7.74,
            "HYG": 2.95,
        },

        "Convexity": {
            "SHY": 0.04,
            "IEF": 0.56,
            "TLT": 3.16,
            "LQD": 1.04,
            "HYG": -0.21,
        },

        "YTM_pct": {
            "SHY": 4.22,
            "IEF": 4.59,
            "TLT": 5.22,
            "LQD": 5.58,
            "HYG": 6.90,
        },

        "OAS_bps": {
            "SHY": -0.62,
            "IEF": 2.98,
            "TLT": 0.12,
            "LQD": 83.95,
            "HYG": 237.77,
        },
    }
)

# Public iShares pages do not provide a separate spread-duration
# measure for LQD and HYG.
# Effective duration is used only as a first-order proxy.
risk_snapshot["Spread_Duration_Proxy"] = 0.0

risk_snapshot.loc[
    ["LQD", "HYG"],
    "Spread_Duration_Proxy"
] = risk_snapshot.loc[
    ["LQD", "HYG"],
    "Effective_Duration"
]

risk_snapshot

,Effective_Duration,Convexity,YTM_pct,OAS_bps,Spread_Duration_Proxy
SHY,1.78,0.04,4.22,-0.62,0.00
IEF,6.82,0.56,4.59,2.98,0.00
TLT,14.95,3.16,5.22,0.12,0.00
LQD,7.74,1.04,5.58,83.95,7.74
HYG,2.95,-0.21,6.90,237.77,2.95


In [4]:
assert (
    risk_snapshot.index.tolist()
    == ETF_TICKERS
)

assert (
    risk_snapshot[
        "Effective_Duration"
    ] > 0
).all()

assert (
    risk_snapshot.loc[
        ["SHY", "IEF", "TLT"],
        "Spread_Duration_Proxy"
    ] == 0
).all()

assert (
    risk_snapshot.loc[
        ["LQD", "HYG"],
        "Spread_Duration_Proxy"
    ] > 0
).all()

assert (
    risk_snapshot.loc[
        "SHY",
        "Effective_Duration"
    ]
    <
    risk_snapshot.loc[
        "IEF",
        "Effective_Duration"
    ]
    <
    risk_snapshot.loc[
        "TLT",
        "Effective_Duration"
    ]
)

print("ETF risk snapshot validation passed.")

ETF risk snapshot validation passed.


## 2. Bond-Level Duration Example

A simple fixed-rate bond is used to demonstrate the difference between:

- Macaulay duration
- Modified duration
- Convexity
- DV01

ETF stress analysis later uses issuer-reported effective duration rather than treating ETF portfolios as single fixed-maturity bonds.

In [5]:
def bond_price(
    face_value,
    coupon_rate,
    ytm,
    years,
    frequency=2,
):
    n_periods = int(
        round(years * frequency)
    )

    periodic_yield = (
        ytm / frequency
    )

    coupon = (
        face_value
        * coupon_rate
        / frequency
    )

    periods = np.arange(
        1,
        n_periods + 1
    )

    cash_flows = np.full(
        n_periods,
        coupon,
        dtype=float,
    )

    cash_flows[-1] += face_value

    discount_factors = (
        1.0
        + periodic_yield
    ) ** periods

    price = (
        cash_flows
        / discount_factors
    ).sum()

    return price


def macaulay_duration(
    face_value,
    coupon_rate,
    ytm,
    years,
    frequency=2,
):
    n_periods = int(
        round(years * frequency)
    )

    periodic_yield = (
        ytm / frequency
    )

    coupon = (
        face_value
        * coupon_rate
        / frequency
    )

    periods = np.arange(
        1,
        n_periods + 1
    )

    cash_flows = np.full(
        n_periods,
        coupon,
        dtype=float,
    )

    cash_flows[-1] += face_value

    discount_factors = (
        1.0
        + periodic_yield
    ) ** periods

    pv_cash_flows = (
        cash_flows
        / discount_factors
    )

    price = (
        pv_cash_flows.sum()
    )

    times_years = (
        periods / frequency
    )

    duration = (
        times_years
        * pv_cash_flows
    ).sum() / price

    return duration


def modified_duration(
    face_value,
    coupon_rate,
    ytm,
    years,
    frequency=2,
):
    mac_duration = macaulay_duration(
        face_value=face_value,
        coupon_rate=coupon_rate,
        ytm=ytm,
        years=years,
        frequency=frequency,
    )

    return (
        mac_duration
        / (
            1.0
            + ytm / frequency
        )
    )


def numerical_convexity(
    face_value,
    coupon_rate,
    ytm,
    years,
    frequency=2,
    yield_shock=0.0001,
):
    price_0 = bond_price(
        face_value,
        coupon_rate,
        ytm,
        years,
        frequency,
    )

    price_down = bond_price(
        face_value,
        coupon_rate,
        ytm - yield_shock,
        years,
        frequency,
    )

    price_up = bond_price(
        face_value,
        coupon_rate,
        ytm + yield_shock,
        years,
        frequency,
    )

    convexity = (
        price_down
        + price_up
        - 2.0 * price_0
    ) / (
        price_0
        * yield_shock ** 2
    )

    return convexity


def calculate_dv01(
    market_value,
    duration,
):
    return (
        market_value
        * duration
        * 0.0001
    )

In [6]:
EXAMPLE_BOND = {
    "face_value": 100,
    "coupon_rate": 0.04,
    "ytm": 0.045,
    "years": 5,
    "frequency": 2,
}

example_price = bond_price(
    **EXAMPLE_BOND
)

example_macaulay = (
    macaulay_duration(
        **EXAMPLE_BOND
    )
)

example_modified = (
    modified_duration(
        **EXAMPLE_BOND
    )
)

example_convexity = (
    numerical_convexity(
        **EXAMPLE_BOND
    )
)

example_dv01 = (
    calculate_dv01(
        market_value=1_000_000,
        duration=example_modified,
    )
)

print("Bond price:")
print(example_price)

print("\nMacaulay duration:")
print(example_macaulay)

print("\nModified duration:")
print(example_modified)

print("\nNumerical convexity:")
print(example_convexity)

print("\nDV01 on $1,000,000 position:")
print(example_dv01)

Bond price:
97.7834459127829

Macaulay duration:
4.575345624627525

Modified duration:
4.474665647557482

Numerical convexity:
23.34623475363592

DV01 on $1,000,000 position:
447.4665647557482


In [7]:
assert example_price > 0

assert (
    example_modified
    <
    example_macaulay
)

assert (
    example_modified > 0
)

assert (
    example_convexity > 0
)

assert (
    example_dv01 > 0
)

print(
    "Bond duration / convexity example validation passed."
)

Bond duration / convexity example validation passed.


In [9]:
PRIMARY_WEIGHTS = {
    "Falling / Tightening": {
        "SHY": 0.10,
        "IEF": 0.15,
        "TLT": 0.25,
        "LQD": 0.30,
        "HYG": 0.20,
    },

    "Falling / Widening": {
        "SHY": 0.10,
        "IEF": 0.30,
        "TLT": 0.40,
        "LQD": 0.20,
        "HYG": 0.00,
    },

    "Rising / Tightening": {
        "SHY": 0.35,
        "IEF": 0.15,
        "TLT": 0.00,
        "LQD": 0.30,
        "HYG": 0.20,
    },

    "Rising / Widening": {
        "SHY": 0.55,
        "IEF": 0.25,
        "TLT": 0.00,
        "LQD": 0.20,
        "HYG": 0.00,
    },

    "Stable / Mixed": {
        "SHY": 0.20,
        "IEF": 0.20,
        "TLT": 0.20,
        "LQD": 0.20,
        "HYG": 0.20,
    },
}

BENCHMARK_WEIGHTS = {
    "Equal Weight": {
        "SHY": 0.20,
        "IEF": 0.20,
        "TLT": 0.20,
        "LQD": 0.20,
        "HYG": 0.20,
    },

    "Core 60/40": {
        "SHY": 0.00,
        "IEF": 0.60,
        "TLT": 0.00,
        "LQD": 0.40,
        "HYG": 0.00,
    },
}

In [10]:
NOTIONAL = 1_000_000


def aggregate_portfolio_risk(
    weights,
    risk_data,
    notional=1_000_000,
):
    weight_series = pd.Series(
        weights,
        dtype=float,
    ).reindex(
        risk_data.index
    )

    effective_duration = (
        weight_series
        * risk_data[
            "Effective_Duration"
        ]
    ).sum()

    convexity = (
        weight_series
        * risk_data[
            "Convexity"
        ]
    ).sum()

    spread_duration_proxy = (
        weight_series
        * risk_data[
            "Spread_Duration_Proxy"
        ]
    ).sum()

    rate_dv01 = (
        notional
        * effective_duration
        * 0.0001
    )

    spread_dv01_proxy = (
        notional
        * spread_duration_proxy
        * 0.0001
    )

    return {
        "Effective_Duration":
            effective_duration,

        "Convexity":
            convexity,

        "Spread_Duration_Proxy":
            spread_duration_proxy,

        "Rate_DV01_per_$1m":
            rate_dv01,

        "Spread_DV01_Proxy_per_$1m":
            spread_dv01_proxy,
    }

In [11]:
portfolio_risk_results = {}

for name, weights in (
    PRIMARY_WEIGHTS.items()
):
    portfolio_risk_results[
        name
    ] = aggregate_portfolio_risk(
        weights,
        risk_snapshot,
        NOTIONAL,
    )

for name, weights in (
    BENCHMARK_WEIGHTS.items()
):
    portfolio_risk_results[
        name
    ] = aggregate_portfolio_risk(
        weights,
        risk_snapshot,
        NOTIONAL,
    )

portfolio_risk_table = (
    pd.DataFrame(
        portfolio_risk_results
    ).T
)

portfolio_risk_table.round(3)

,Effective_Duration,Convexity,Spread_Duration_Proxy,Rate_DV01_per_$1m,Spread_DV01_Proxy_per_$1m
Falling / Tightening,7.850,1.148,2.912,785.05,291.2
Falling / Widening,9.752,1.644,1.548,975.20,154.8
Rising / Tightening,4.558,0.368,2.912,455.80,291.2
Rising / Widening,4.232,0.370,1.548,423.20,154.8
Stable / Mixed,6.848,0.918,2.138,684.80,213.8
Equal Weight,6.848,0.918,2.138,684.80,213.8
Core 60/40,7.188,0.752,3.096,718.80,309.6


In [12]:
assert (
    portfolio_risk_table.loc[
        "Falling / Widening",
        "Effective_Duration"
    ]
    >
    portfolio_risk_table.loc[
        "Rising / Widening",
        "Effective_Duration"
    ]
)

assert (
    portfolio_risk_table.loc[
        "Falling / Tightening",
        "Effective_Duration"
    ]
    >
    portfolio_risk_table.loc[
        "Rising / Tightening",
        "Effective_Duration"
    ]
)

assert (
    portfolio_risk_table.loc[
        "Rising / Tightening",
        "Spread_Duration_Proxy"
    ]
    >
    portfolio_risk_table.loc[
        "Rising / Widening",
        "Spread_Duration_Proxy"
    ]
)

print(
    "Portfolio risk aggregation validation passed."
)

Portfolio risk aggregation validation passed.


## 3. Parallel Rate-Shock Scenarios

Interest-rate sensitivity is approximated using issuer-reported effective duration:

\[
\Delta P / P \approx -D_{eff}\Delta y
\]

The scenarios are:

- +50 bp parallel rate shock
- +100 bp parallel rate shock

Convexity is shown separately as a portfolio characteristic rather than forced into the ETF scenario calculation because the public fund pages do not provide sufficient scaling detail for direct second-order repricing.

In [13]:
RATE_SHOCKS_BPS = [
    50,
    100,
]

rate_shock_returns = pd.DataFrame(
    index=ETF_TICKERS
)

for shock_bps in RATE_SHOCKS_BPS:

    rate_shock_returns[
        f"Rates_+{shock_bps}bp"
    ] = (
        -risk_snapshot[
            "Effective_Duration"
        ]
        * shock_bps
        / 10_000
    )

rate_shock_returns

,Rates_+50bp,Rates_+100bp
SHY,-0.00890,-0.0178
IEF,-0.03410,-0.0682
TLT,-0.07475,-0.1495
LQD,-0.03870,-0.0774
HYG,-0.01475,-0.0295


## 4. Credit-Spread Shock Scenarios

Credit shocks are applied only to LQD and HYG.

Because public iShares fund characteristics do not provide a standalone spread-duration measure, effective duration is used as a first-order spread-duration proxy:

\[
\Delta P / P
\approx
-D_{spread,proxy}\Delta s
\]

The scenarios are:

- +100 bp credit-spread shock
- +300 bp credit-spread shock

These results are sensitivity estimates, not exact portfolio repricing forecasts.

In [14]:
CREDIT_SHOCKS_BPS = [
    100,
    300,
]

credit_shock_returns = pd.DataFrame(
    index=ETF_TICKERS
)

for shock_bps in CREDIT_SHOCKS_BPS:

    credit_shock_returns[
        f"Credit_+{shock_bps}bp"
    ] = (
        -risk_snapshot[
            "Spread_Duration_Proxy"
        ]
        * shock_bps
        / 10_000
    )

credit_shock_returns

,Credit_+100bp,Credit_+300bp
SHY,-0.0000,-0.0000
IEF,-0.0000,-0.0000
TLT,-0.0000,-0.0000
LQD,-0.0774,-0.2322
HYG,-0.0295,-0.0885


In [15]:
assert np.isclose(
    rate_shock_returns.loc[
        "TLT",
        "Rates_+100bp"
    ],
    -0.1495,
)

assert (
    credit_shock_returns.loc[
        ["SHY", "IEF", "TLT"]
    ] == 0
).all().all()

assert (
    credit_shock_returns.loc[
        "LQD",
        "Credit_+300bp"
    ]
    <
    credit_shock_returns.loc[
        "LQD",
        "Credit_+100bp"
    ]
)

print(
    "ETF scenario validation passed."
)

ETF scenario validation passed.


In [16]:
def aggregate_scenario_return(
    weights,
    asset_scenario_returns,
):
    weight_series = pd.Series(
        weights,
        dtype=float,
    ).reindex(
        asset_scenario_returns.index
    )

    return (
        asset_scenario_returns
        .mul(
            weight_series,
            axis=0,
        )
        .sum(axis=0)
    )

In [17]:
combined_asset_scenarios = (
    rate_shock_returns
    .join(
        credit_shock_returns
    )
)

portfolio_scenario_results = {}

for name, weights in {
    **PRIMARY_WEIGHTS,
    **BENCHMARK_WEIGHTS,
}.items():

    portfolio_scenario_results[
        name
    ] = aggregate_scenario_return(
        weights,
        combined_asset_scenarios,
    )

portfolio_scenario_table = (
    pd.DataFrame(
        portfolio_scenario_results
    ).T
)

portfolio_scenario_display = (
    portfolio_scenario_table
    * 100
)

portfolio_scenario_display.round(2)

,Rates_+50bp,Rates_+100bp,Credit_+100bp,Credit_+300bp
Falling / Tightening,-3.93,-7.85,-2.91,-8.74
Falling / Widening,-4.88,-9.75,-1.55,-4.64
Rising / Tightening,-2.28,-4.56,-2.91,-8.74
Rising / Widening,-2.12,-4.23,-1.55,-4.64
Stable / Mixed,-3.42,-6.85,-2.14,-6.41
Equal Weight,-3.42,-6.85,-2.14,-6.41
Core 60/40,-3.59,-7.19,-3.10,-9.29


In [18]:
latest_date = (
    regime_data.index.max()
)

latest_regime = (
    regime_data.loc[
        latest_date,
        "Regime"
    ]
)

print(
    "Latest signal date:",
    latest_date
)

print(
    "Latest observed regime:",
    latest_regime
)

print(
    "\nNext-month target weights:"
)

latest_target_weights = (
    PRIMARY_WEIGHTS[
        latest_regime
    ]
)

for ticker, weight in (
    latest_target_weights.items()
):
    print(
        ticker,
        f"{weight:.0%}"
    )

Latest signal date: 2026-07-31 00:00:00
Latest observed regime: Rising / Tightening

Next-month target weights:
SHY 35%
IEF 15%
TLT 0%
LQD 30%
HYG 20%


In [19]:
latest_portfolio_risk = (
    aggregate_portfolio_risk(
        latest_target_weights,
        risk_snapshot,
        NOTIONAL,
    )
)

latest_scenario_returns = (
    aggregate_scenario_return(
        latest_target_weights,
        combined_asset_scenarios,
    )
)

print(
    "Latest regime:",
    latest_regime
)

print(
    "\nPortfolio risk:"
)

for key, value in (
    latest_portfolio_risk.items()
):
    print(
        key,
        ":",
        value
    )

print(
    "\nScenario returns:"
)

for scenario, value in (
    latest_scenario_returns.items()
):
    print(
        scenario,
        ":",
        f"{value:.2%}"
    )

Latest regime: Rising / Tightening

Portfolio risk:
Effective_Duration : 4.558
Convexity : 0.36800000000000005
Spread_Duration_Proxy : 2.912
Rate_DV01_per_$1m : 455.8
Spread_DV01_Proxy_per_$1m : 291.2

Scenario returns:
Rates_+50bp : -2.28%
Rates_+100bp : -4.56%
Credit_+100bp : -2.91%
Credit_+300bp : -8.74%


In [20]:
latest_weight_series = (
    pd.Series(
        latest_target_weights
    )
    .reindex(
        ETF_TICKERS
    )
)

latest_risk_contribution = pd.DataFrame(
    index=ETF_TICKERS
)

latest_risk_contribution[
    "Weight"
] = latest_weight_series

latest_risk_contribution[
    "Effective_Duration"
] = risk_snapshot[
    "Effective_Duration"
]

latest_risk_contribution[
    "Rate_DV01_Contribution"
] = (
    NOTIONAL
    * latest_risk_contribution[
        "Weight"
    ]
    * latest_risk_contribution[
        "Effective_Duration"
    ]
    * 0.0001
)

latest_risk_contribution[
    "Spread_DV01_Proxy_Contribution"
] = (
    NOTIONAL
    * latest_risk_contribution[
        "Weight"
    ]
    * risk_snapshot[
        "Spread_Duration_Proxy"
    ]
    * 0.0001
)

latest_risk_contribution.round(2)

,Weight,Effective_Duration,Rate_DV01_Contribution,Spread_DV01_Proxy_Contribution
SHY,0.35,1.78,62.3,0.0
IEF,0.15,6.82,102.3,0.0
TLT,0.00,14.95,0.0,0.0
LQD,0.30,7.74,232.2,232.2
HYG,0.20,2.95,59.0,59.0


In [21]:
assert np.isclose(
    latest_risk_contribution[
        "Rate_DV01_Contribution"
    ].sum(),
    latest_portfolio_risk[
        "Rate_DV01_per_$1m"
    ],
)

assert np.isclose(
    latest_risk_contribution[
        "Spread_DV01_Proxy_Contribution"
    ].sum(),
    latest_portfolio_risk[
        "Spread_DV01_Proxy_per_$1m"
    ],
)

print(
    "Risk contribution validation passed."
)

Risk contribution validation passed.


In [22]:
RISK_TABLE_PATH = (
    PROCESSED_DIR
    / "fixed_income_risk_snapshot.csv"
)

PORTFOLIO_RISK_PATH = (
    PROCESSED_DIR
    / "fixed_income_portfolio_risk.csv"
)

SCENARIO_PATH = (
    PROCESSED_DIR
    / "fixed_income_scenario_results.csv"
)

risk_snapshot.to_csv(
    RISK_TABLE_PATH
)

portfolio_risk_table.to_csv(
    PORTFOLIO_RISK_PATH
)

portfolio_scenario_table.to_csv(
    SCENARIO_PATH
)

print(
    "Risk snapshot:",
    RISK_TABLE_PATH.exists()
)

print(
    "Portfolio risk:",
    PORTFOLIO_RISK_PATH.exists()
)

print(
    "Scenario results:",
    SCENARIO_PATH.exists()
)

Risk snapshot: True
Portfolio risk: True
Scenario results: True


In [23]:
assert (
    latest_regime
    in PRIMARY_WEIGHTS
)

assert (
    latest_portfolio_risk[
        "Effective_Duration"
    ] > 0
)

assert (
    latest_portfolio_risk[
        "Rate_DV01_per_$1m"
    ] > 0
)

assert (
    portfolio_scenario_table[
        "Rates_+100bp"
    ]
    <= 0
).all()

assert (
    portfolio_scenario_table[
        "Credit_+300bp"
    ]
    <= 0
).all()

print("=" * 60)

print(
    "STEP 6 — FIXED INCOME RISK & SCENARIO ANALYSIS COMPLETE"
)

print("=" * 60)

print(
    "\nETF duration source:",
    "iShares portfolio characteristics"
)

print(
    "Risk snapshot date:",
    "2026-08-27"
)

print(
    "\nLatest signal date:",
    latest_date.date()
)

print(
    "Latest regime:",
    latest_regime
)

print(
    "\nRate scenarios:",
    "+50bp / +100bp"
)

print(
    "Credit scenarios:",
    "+100bp / +300bp"
)

print(
    "\nCredit spread duration is explicitly treated "
    "as an approximation."
)

STEP 6 — FIXED INCOME RISK & SCENARIO ANALYSIS COMPLETE

ETF duration source: iShares portfolio characteristics
Risk snapshot date: 2026-08-27

Latest signal date: 2026-07-31
Latest regime: Rising / Tightening

Rate scenarios: +50bp / +100bp
Credit scenarios: +100bp / +300bp

Credit spread duration is explicitly treated as an approximation.


## Step 6 Conclusion

The fixed-income universe and frozen regime portfolios have been translated into explicit interest-rate and credit-risk exposures.

The analysis distinguishes:

- bond-level Macaulay and modified duration,
- issuer-reported ETF effective duration,
- portfolio DV01,
- convexity,
- rate-shock sensitivity,
- and approximate credit-spread sensitivity.

Credit-spread stress uses effective duration only as a first-order spread-duration proxy and is therefore not presented as exact bond-level repricing.

The next stage decomposes observed strategy performance into rate, credit, and residual/carry components.